In [15]:
 
import os
import warnings
import joblib
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans, DBSCAN
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score
)
from sklearn.preprocessing import normalize
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from wordcloud import WordCloud

In [17]:
# CONFIGURATION
 
CSV_PATH               = "../data/cleaned_enron_v6.csv"
N_CLUSTERS             = 10     
TFIDF_FEATURES         = 10000  
TEST_SIZE              = 0.20    
RANDOM_STATE           = 42
CONFIDENCE_THRESHOLD   = 0.70    
MIN_WORDS_TO_CLASSIFY  = 3      
OUTPUT_PKL             = "email_priority_detector.pkl"

In [14]:
#  PART 1  —  LOAD DATA  +  PREPROCESSING
print("\n" + "=" * 65)
print("  PART 1  —  Load Data  +  Preprocessing")
print("=" * 65)
 
df = pd.read_csv(CSV_PATH)
import re

def clean_email_text(text):
    text = str(text).lower()

    # Remove time expressions
    text = re.sub(r'\b\d{1,2}:\d{2}\b', ' ', text)
    text = re.sub(r'\b\d{1,2}(am|pm)\b', ' ', text)

    # Remove business/energy codes
    text = re.sub(r'\b\d+\b', ' ', text)
    text = re.sub(r'\b(mmbtu|ect|hou|corp|dth|cst|cdt|pst|est)\b', ' ', text)

    # Remove legal identifiers
    text = re.sub(r'\b(isda|naesb|ferc|section|article|agreement number)\b', ' ', text)

    # Remove punctuation leftovers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df = df[['subject', 'body']].fillna('')
 
df['text'] = ((df['subject'] + ' ') * 3 + df['body']).apply(clean_email_text)
df = df[df['text'].str.strip().str.len() > 10].reset_index(drop=True)
 
print(f"  Dataset : {CSV_PATH}")
print(f"  Total emails loaded  : {len(df):,}")
print(f"  Preprocessing rule   : (subject × 3) + body")
print(f"\n  Sample — email #1 text (first 200 chars):")
print(f"  {df['text'].iloc[0][:200]}")


  PART 1  —  Load Data  +  Preprocessing
  Dataset : ../data/cleaned_enron_v6.csv
  Total emails loaded  : 115,773
  Preprocessing rule   : (subject × 3) + body

  Sample — email #1 text (first 200 chars):
  no subject no subject no subject travel business meeting take fun trip especially prepare presentation would suggest hold business plan meeting take trip without formal business meeting would even try


In [18]:
#  PART 1.5  —  EXPLORATORY DATA ANALYSIS  (EDA)
print("\n" + "=" * 65)
print("  PART 1.5  —  Exploratory Data Analysis")
print("=" * 65)
 
df['email_length'] = df['text'].str.len() # Email length distribution
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
axes[0].hist(df['email_length'].clip(upper=3000), bins=60,
             color='#2471A3', edgecolor='white', linewidth=0.4)
axes[0].set_title("Email Length Distribution  (clipped at 3000 chars)",
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Number of Emails")
 
axes[1].boxplot(df['email_length'].clip(upper=3000),
                vert=False, patch_artist=True,
                boxprops=dict(facecolor='#AED6F1'))
axes[1].set_title("Email Length — Box Plot", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Characters")
 
plt.suptitle(f"Enron Dataset — {len(df):,} Emails", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("V3_email_length_distribution.png", dpi=130)
plt.close()
print("  Saved: email_length_distribution.png")
 
print(f"  Length stats:")
print(f"    Min    : {df['email_length'].min()}")
print(f"    Median : {df['email_length'].median():.0f}")
print(f"    Mean   : {df['email_length'].mean():.0f}")
print(f"    Max    : {df['email_length'].max()}")
 
# Global word cloud 
print("  Generating global word cloud (50k sample)...")
sample_text = " ".join(
    df['text'].sample(min(50000, len(df)), random_state=RANDOM_STATE)
)
wc = WordCloud(
    width=1600, height=800,
    background_color='white',
    max_words=300,
    colormap='Blues',
    collocations=False
).generate(sample_text)
 
fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title("Global Word Cloud — All 115,773 Enron Emails",
             fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig("V3_wordcloud_global.png", dpi=130)
plt.close()
print("  Saved: wordcloud_global.png")


  PART 1.5  —  Exploratory Data Analysis
  Saved: email_length_distribution.png
  Length stats:
    Min    : 36
    Median : 270
    Mean   : 408
    Max    : 26350
  Generating global word cloud (50k sample)...
  Saved: wordcloud_global.png


In [ ]:
#  PART 2  —  TF-IDF FEATURE EXTRACTION
print("\n" + "=" * 65)
print("  PART 2  —  TF-IDF Feature Extraction")
print("=" * 65)

 
vectorizer = TfidfVectorizer(
    max_features = TFIDF_FEATURES,   # keep the 10,000 most informative terms
    min_df       = 8,                # ignore terms in fewer than 8 emails
    max_df       = 0.85,             # ignore terms in 85 %+ of all emails
    stop_words   = 'english',        # remove: "the", "is", "and", etc.
    ngram_range  = (1, 2),
    token_pattern=r'\b[a-zA-Z]{3,}\b' # single words AND two-word phrases
    stop_words=list(ENGLISH_STOP_WORDS.union(custom_noise))

)
 
X = vectorizer.fit_transform(df['text'])
feature_names = vectorizer.get_feature_names_out()
 
print(f"  TF-IDF matrix shape  : {X.shape[0]:,} emails  ×  {X.shape[1]:,} features")
print(f"  Non-zero entries     : {X.nnz:,}  (sparse — memory efficient)")
print(f"  Sample vocabulary    : {list(feature_names[:20])}")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2039343035.py, line 17)

In [ ]:
#  PART 3A  —  K-MEANS CLUSTERING  (Primary unsupervised step)
